<h1>Frequency Domain Steganography</h1>

### Necessary Imports & Declarations

In [ ]:
import os
import glob
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
# matplotlib inline

from PIL import Image
from torchvision import transforms
from skimage.filters import threshold_otsu

cv2.saliency

# Define directory paths
train_dir = os.path.join("data", "imagenet", "train")
val_dir   = os.path.join("data", "imagenet", "val")

# ADD THIS — Bossbase grayscale image dir
bossbase_dir = os.path.join("data", "bossbase")


### Preview function to see output of each step.

In [ ]:
def get_grayscale_images(directory, n=10):
    image_paths = glob.glob(os.path.join(directory, '*.*'))
    if len(image_paths) < n:
        image_paths = image_paths
    else:
        image_paths = random.sample(image_paths, n)

    images = []
    titles = []

    for path in image_paths:
        try:
            img = Image.open(path).convert("L")  # Convert to grayscale
            img = img.resize((224, 224))
            images.append(np.array(img) / 255.0)  # Normalize
            titles.append(os.path.basename(path))
        except Exception as e:
            print(f"Error loading {path}: {e}")
    
    return images, titles

# Load grayscale Bossbase images
gray_images, gray_titles = get_grayscale_images(bossbase_dir, n=10)


### Load and Preprocess Images

In [ ]:
# Display the resized grayscale images horizontally
fig, axes = plt.subplots(1, len(gray_images), figsize=(len(gray_images) * 4, 4))
for ax, img, title in zip(axes, gray_images, gray_titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


### DeepGaze Saliency Mapping

In [ ]:
def gen_saliency_maps_grayscale(images):
    saliency_maps = []
    saliency_detector = cv2.saliency.StaticSaliencyFineGrained_create()

    for idx, img in enumerate(images):
        image_uint8 = (img * 255).astype(np.uint8)
        success, sal_map = saliency_detector.computeSaliency(image_uint8)

        if not success:
            print(f"Saliency failed for image {idx+1}")
            continue

        sal_map = (sal_map * 255).astype(np.uint8)
        saliency_maps.append(sal_map)
        print(f"Processed image {idx+1}: shape {image_uint8.shape}, min {image_uint8.min()}, max {image_uint8.max()}")

    return saliency_maps

# Generate saliency maps for grayscale images
gray_saliency_maps = gen_saliency_maps_grayscale(gray_images)

# Otsu threshold & visualization
gray_binary_masks = []
fig, axes = plt.subplots(3, len(gray_images), figsize=(len(gray_images) * 4, 12))

for i, (img, sal_map) in enumerate(zip(gray_images, gray_saliency_maps)):
    sal_map_norm = cv2.normalize(sal_map, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
    thresh_val = threshold_otsu(sal_map_norm)
    binary_mask = (sal_map_norm < thresh_val).astype(np.uint8) * 255
    gray_binary_masks.append(binary_mask)

    # Row 0: Grayscale input
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f"Original: {gray_titles[i]}")
    axes[0, i].axis('off')

    # Row 1: Saliency Map
    axes[1, i].imshow(sal_map_norm, cmap='hot')
    axes[1, i].set_title(f"Saliency Map {i+1}")
    axes[1, i].axis('off')

    # Row 2: Binary Mask
    axes[2, i].imshow(binary_mask, cmap='gray')
    axes[2, i].set_title(f"Binary Mask {i+1}")
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()


<h2>Frequency Domain Embedding (DCT)</h2>

### Gnereate the block mask

In [ ]:
def compute_block_mask(saliency_map, block_size=16, threshold=0.05):
    """
    Compute a binary mask indicating which blocks are good for embedding
    based on average saliency per block.

    Args:
        saliency_map (numpy array): Grayscale saliency map (HxW), normalized [0,1]
        block_size (int): Block size, default 16
        threshold (float): Saliency threshold

    Returns:
        mask (numpy array): Binary mask (h_blocks x w_blocks), 1=good for embedding, 0=skip
    """
    h, w = saliency_map.shape
    h_blocks = h // block_size
    w_blocks = w // block_size

    mask = np.zeros((h_blocks, w_blocks), dtype=np.uint8)

    for i in range(h_blocks):
        for j in range(w_blocks):
            y_start = i * block_size
            x_start = j * block_size
            block = saliency_map[y_start:y_start + block_size, x_start:x_start + block_size]
            avg_saliency = np.mean(block)

            if avg_saliency < threshold:
                mask[i, j] = 1  # Good block for embedding

    return mask


### Overlay the Mask on the Image

In [ ]:
def overlay_blocks_on_image(image, block_mask, block_size=16):
    """
    Overlay green/red rectangles on the original grayscale image based on the block mask.

    Args:
        image (numpy array): Grayscale image (HxW), uint8
        block_mask (numpy array): Binary block mask (h_blocks x w_blocks)
        block_size (int): Block size, default 16

    Returns:
        overlay_img (numpy array): Grayscale image with green/red block overlays (HxWx3)
    """
    # Convert grayscale to BGR so we can draw colored rectangles
    if len(image.shape) == 2:
        overlay_img = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        overlay_img = image.copy()

    h_blocks, w_blocks = block_mask.shape

    for i in range(h_blocks):
        for j in range(w_blocks):
            y_start = i * block_size
            x_start = j * block_size

            color = (0, 255, 0) if block_mask[i, j] == 1 else (255, 0, 0)

            cv2.rectangle(
                overlay_img,
                (x_start, y_start),
                (x_start + block_size - 1, y_start + block_size - 1),
                color,
                1
            )

    return overlay_img


### Display Saliency Maps and Block Overlays

In [ ]:
def display_saliency_and_overlay(saliency_maps, overlay_images, titles=None):
    """
    Display saliency maps and overlay images in two rows:
    - Row 0: Saliency maps (hot colormap)
    - Row 1: Overlay images with green/red blocks

    Args:
        saliency_maps (list): List of saliency maps (HxW), normalized [0,1] or [0,255]
        overlay_images (list): List of overlay images (HxWx3), uint8
        titles (list): Optional list of titles for each image
    """
    num_images = len(saliency_maps)
    fig, axes = plt.subplots(2, num_images, figsize=(num_images * 4, 8))

    for i in range(num_images):
        sal_map = saliency_maps[i]
        overlay_img = overlay_images[i]

        # Normalize saliency map for display
        if sal_map.max() <= 1.0:
            sal_map_vis = (sal_map * 255).astype(np.uint8)
        else:
            sal_map_vis = sal_map.astype(np.uint8)

        # Saliency map
        axes[0, i].imshow(sal_map_vis, cmap='hot')
        axes[0, i].set_title(f"Saliency Map {i+1}" if not titles else titles[i])
        axes[0, i].axis('off')

        # Overlay image
        axes[1, i].imshow(overlay_img)
        axes[1, i].set_title(f"Block Overlay {i+1}" if not titles else titles[i])
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()

    # Parameters
block_size = 16
threshold = 0.05

# Containers for masks and overlay images
gray_block_masks = []
gray_overlay_images = []

# Process each grayscale image and its saliency map
for img, sal_map in zip(gray_images, gray_saliency_maps):
    img_uint8 = (img * 255).astype(np.uint8)
    sal_map_norm = sal_map / 255.0 if sal_map.max() > 1.0 else sal_map

    # Step 1: Compute block mask
    block_mask = compute_block_mask(sal_map_norm, block_size, threshold)
    gray_block_masks.append(block_mask)

    # Step 2: Overlay block mask on image
    overlay_img = overlay_blocks_on_image(img_uint8, block_mask, block_size)
    gray_overlay_images.append(overlay_img)

# Step 3: Display results
display_saliency_and_overlay(gray_saliency_maps, gray_overlay_images, titles=gray_titles)


### Calculate Capacity

In [ ]:
def calculate_embedding_capacity(block_mask, bits_per_block=4):
    """
    Calculate embedding capacity based on block mask and bits per block.
    
    Args:
        block_mask (numpy array): Binary mask of blocks (h_blocks x w_blocks)
        bits_per_block (int): Number of bits you can embed per block (default: 4)
    
    Returns:
        total_bits (int): Total embedding capacity in bits
    """
    num_good_blocks = np.sum(block_mask == 1)
    total_bits = num_good_blocks * bits_per_block
    print(f"Embedding capacity: {total_bits} bits ({num_good_blocks} blocks x {bits_per_block} bits/block)")
    return total_bits



### Generate Random Payload Bits

In [ ]:
def generate_random_payload(num_bits):
    """
    Generate a random payload as a bitstream.
    
    Args:
        num_bits (int): Number of bits to generate
    
    Returns:
        payload_bits (numpy array): Array of 0s and 1s
    """
    payload_bits = np.random.randint(0, 2, size=num_bits, dtype=np.uint8)
    print(f"Generated random payload of {num_bits} bits.")
    return payload_bits


### Convert String Message to Bits

In [ ]:
def string_to_bits(message):
    """
    Convert a string message to a bitstream (list of 0s and 1s).
    
    Args:
        message (str): Text message to convert
    
    Returns:
        payload_bits (numpy array): Bitstream of the message
    """
    byte_array = bytearray(message, 'utf-8')
    bits = []
    for byte in byte_array:
        bits.extend([int(bit) for bit in format(byte, '08b')])
    
    payload_bits = np.array(bits, dtype=np.uint8)
    print(f"Converted message '{message}' to {len(payload_bits)} bits.")
    return payload_bits


In [ ]:
# Step 1: Calculate capacity for each grayscale image
bits_per_block = 4  # Or adjust this if needed
capacities = [calculate_embedding_capacity(mask, bits_per_block=bits_per_block) for mask in gray_block_masks]

# Step 2a: Generate random payloads for each grayscale image
random_payloads = [generate_random_payload(capacity) for capacity in capacities]

# Step 2b: Optional — Convert a string to bitstream
message = "War Eagle!"
message_payload = string_to_bits(message)

# Step 3: Fit message payload into each image (if possible)
payloads = []
for capacity in capacities:
    if len(message_payload) > capacity:
        print("Message is too big! Trimming it to fit.")
        payload = message_payload[:capacity]
    else:
        payload = message_payload
    payloads.append(payload)


## STEP 6 - Qunatization tables for luminance and chrominance

### Extract JPEG Quantization Tables (QnT)

In [ ]:
def extract_quantization_tables(image_paths):
    """
    Extract JPEG quantization tables from image files.

    Args:
        image_paths (list): List of file paths to images.

    Returns:
        qtables_list (list): List of quantization tables for each image.
    """
    qtables_list = []

    for idx, img_path in enumerate(image_paths):
        try:
            img = Image.open(img_path)

            # Check if it's JPEG
            if img.format != 'JPEG':
                print(f"[{idx+1}] {os.path.basename(img_path)} is not a JPEG image. Skipping.")
                qtables_list.append(None)
                continue

            qtables = img.quantization
            qtables_list.append(qtables)

            print(f"[{idx+1}] Extracted quantization tables from {os.path.basename(img_path)}")

        except Exception as e:
            print(f"[{idx+1}] Error processing {img_path}: {e}")
            qtables_list.append(None)

    return qtables_list

# Run it on your sample image paths
qtables_list = extract_quantization_tables(sample_image_paths)

Visualize the QnT Tables (for Understanding)

In [ ]:
def display_quantization_tables(qtables_list, image_titles):
    """
    Display quantization tables for each image in a readable format.

    Args:
        qtables_list (list): List of quantization tables.
        image_titles (list): Corresponding titles or filenames.
    """
    for idx, qtables in enumerate(qtables_list):
        print(f"\n=== Quantization Tables for Image {idx+1}: {image_titles[idx]} ===")
        
        if qtables is None:
            print("No quantization table extracted.")
            continue

        for table_id, qtable in qtables.items():
            qtable_matrix = np.array(qtable).reshape((8,8))
            
            df = pd.DataFrame(qtable_matrix)
            print(f"\nTable {table_id} ({'Luminance' if table_id == 0 else 'Chrominance'}):")
            display(df)

# Visualize the quantization tables for understanding
display_quantization_tables(qtables_list, titles)

## Step 7 - Convert "Good" 16x16 Blocks into DCT Domain
Take each 16x16 "good" block (from your block_masks)
Split it into four 8x8 sub-blocks (because DCT is done in 8x8 blocks)
Perform 2D DCT on each sub-block
Store those DCT coefficients for embedding in the next step


### -  Code Block #1: Helper Function to Perform 2D DCT

In [ ]:
def block_to_dct_blocks(block_16x16):
    """
    Split a 16x16 block into four 8x8 sub-blocks and perform DCT on each.
    
    Args:
        block_16x16 (numpy array): 16x16x3 block (RGB).
    
    Returns:
        dct_blocks (list): List of 4 DCT blocks (each 8x8x3).
    """
    dct_blocks = []

    # Split the 16x16 block into four 8x8 blocks
    for i in range(2):  # rows
        for j in range(2):  # columns
            y_start = i * 8
            x_start = j * 8

            sub_block = block_16x16[y_start:y_start+8, x_start:x_start+8, :]

            # Perform DCT on each channel separately
            dct_sub_block = np.zeros_like(sub_block, dtype=np.float32)
            for c in range(3):  # R, G, B channels
                dct_sub_block[:, :, c] = cv2.dct(sub_block[:, :, c].astype(np.float32))

            dct_blocks.append(dct_sub_block)

    return dct_blocks


### - Code Block #2: Apply to All Images and Good Blocks

In [ ]:
def extract_dct_blocks_from_good_blocks(images, block_masks, block_size=16):
    """
    Extract DCT blocks (4x 8x8 sub-blocks) from each good 16x16 block in all images.
    
    Args:
        images (list): List of sample images (normalized RGB arrays).
        block_masks (list): List of block masks (1 = good for embedding).
        block_size (int): Size of blocks (default 16).
    
    Returns:
        dct_blocks_per_image (list): For each image, a list of DCT blocks.
    """
    dct_blocks_per_image = []

    for img_idx, (img, block_mask) in enumerate(zip(images, block_masks)):
        print(f"\nProcessing image {img_idx+1}/{len(images)}...")
        
        # Ensure image is in uint8
        img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img

        h_blocks, w_blocks = block_mask.shape
        dct_blocks_for_this_image = []

        for i in range(h_blocks):
            for j in range(w_blocks):
                if block_mask[i, j] == 1:  # Good block!
                    y_start = i * block_size
                    x_start = j * block_size

                    # Extract the 16x16 block (HxWxC)
                    block_16x16 = img_uint8[y_start:y_start+block_size, x_start:x_start+block_size, :]

                    # Get the four 8x8 DCT blocks
                    dct_blocks = block_to_dct_blocks(block_16x16)

                    # Append to our list
                    dct_blocks_for_this_image.append({
                        "block_position": (i, j),
                        "dct_blocks": dct_blocks
                    })

        print(f"Extracted {len(dct_blocks_for_this_image)} good 16x16 blocks from image {img_idx+1}")
        dct_blocks_per_image.append(dct_blocks_for_this_image)

    return dct_blocks_per_image

# Run it on your images and block masks
dct_blocks_per_image = extract_dct_blocks_from_good_blocks(sample_images, block_masks)


### - Sanity Check & Visualize DCT Coefficients

In [ ]:
def visualize_dct_block(dct_block, title_prefix="DCT Block"):
    """
    Visualize the magnitude spectrum of a single DCT 8x8 block (RGB channels).
    
    Args:
        dct_block (numpy array): 8x8x3 DCT coefficients.
    """
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for c in range(3):
        # Take absolute value and log scale for visualization
        dct_magnitude = np.log(np.abs(dct_block[:, :, c]) + 1)

        axes[c].imshow(dct_magnitude, cmap='gray')
        axes[c].set_title(f"{title_prefix} - Channel {c} (RGB)")
        axes[c].axis('off')

    plt.tight_layout()
    plt.show()

# Example: Visualize first DCT block of the first image
if dct_blocks_per_image and len(dct_blocks_per_image[0]) > 0:
    example_dct_block = dct_blocks_per_image[0][0]["dct_blocks"][0]  # First 8x8 DCT block of first good 16x16 block
    visualize_dct_block(example_dct_block, title_prefix="Example DCT Block")
else:
    print("No DCT blocks found to visualize.")


## Step 8: Select Mid-Mid DCT Frequencies for Embedding

- From each **8x8 DCT block** inside your good **16x16 blocks**,  
  we select specific **mid-frequency coefficients** to embed your data.

- We'll **adaptively embed**, considering the **QnT values** we extracted earlier (from Step 6).

---

#### **What We’ll Do**

1. Define a list of **mid-mid frequency positions**  
2. For each DCT block, **select those positions**  
3. Apply **QnT-based decision logic**:
    - Skip **low Q** (highly sensitive)  
    - Allow changes in **mid/high Q**  
4. Prepare to **embed bits** in these coefficients in the next step.


### Code Block #1: Define Mid-Mid Frequencies

In [ ]:
# Mid-mid frequency positions (Y-axis first)
mid_mid_frequencies = [(3, 3), (2, 3), (3, 2), (4, 1), (1, 4)]


### Code Block #2: Adaptive Embedding Decision Function

In [ ]:
def should_embed(q_value, q_threshold_low=4, q_threshold_high=8):
    """
    Decide whether to embed based on quantization value.

    Args:
        q_value (int): Quantization table value at that frequency.
        q_threshold_low (int): Lower threshold (too sensitive to modify).
        q_threshold_high (int): Upper threshold (safe for stronger changes).

    Returns:
        bool: True if we should embed here, False otherwise.
    """
    if q_value <= q_threshold_low:
        return False  # Too sensitive, skip
    elif q_value <= q_threshold_high:
        return True   # Safe for minimal embedding (±1)
    else:
        return True   # Safe for stronger embedding (±1 or ±2)


### Code Block #3: Select Embedding Positions in DCT Blocks


In [ ]:
def select_embedding_positions(dct_blocks_per_image, qtables_list, mid_mid_frequencies):
    """
    Select positions in DCT blocks where we can embed data.

    Args:
        dct_blocks_per_image (list): Output from Step 7.
        qtables_list (list): List of quantization tables from Step 6.
        mid_mid_frequencies (list): List of (row, col) tuples of candidate frequencies.

    Returns:
        embedding_candidates_per_image (list): For each image, a list of embedding positions.
    """
    embedding_candidates_per_image = []

    for img_idx, (dct_blocks_info, qtables) in enumerate(zip(dct_blocks_per_image, qtables_list)):
        print(f"\nSelecting embedding positions for Image {img_idx+1}")

        if qtables is None:
            print("No QnT found for this image. Skipping.")
            embedding_candidates_per_image.append([])
            continue

        # Assume luminance Q-table (Table 0) for now
        qtable = np.array(qtables[0]).reshape((8, 8))
        candidates_for_image = []

        for block_info in dct_blocks_info:
            block_position = block_info['block_position']
            dct_blocks = block_info['dct_blocks']

            candidates_for_block = []

            # Iterate over 4 sub-blocks (each 8x8)
            for subblock_idx, dct_block in enumerate(dct_blocks):
                candidates_for_subblock = []

                for (y, x) in mid_mid_frequencies:
                    q_value = qtable[y, x]

                    if should_embed(q_value):
                        candidates_for_subblock.append({
                            "subblock_idx": subblock_idx,
                            "freq": (y, x),
                            "q_value": q_value
                        })

                if candidates_for_subblock:
                    candidates_for_block.append({
                        "block_position": block_position,
                        "subblock_idx": subblock_idx,
                        "candidates": candidates_for_subblock
                    })

            if candidates_for_block:
                candidates_for_image.append(candidates_for_block)

        print(f"Found {len(candidates_for_image)} blocks with embedding opportunities in Image {img_idx+1}")
        embedding_candidates_per_image.append(candidates_for_image)

    return embedding_candidates_per_image

# Run the candidate selection
embedding_candidates_per_image = select_embedding_positions(
    dct_blocks_per_image,
    qtables_list,
    mid_mid_frequencies
)


### Code Block #4: Filter Images With Opportunities

In [ ]:
def filter_images_with_candidates(embedding_candidates_per_image, dct_blocks_per_image, sample_image_paths, titles, payloads):
    """
    Filters out images that have no embedding candidates.

    Args:
        embedding_candidates_per_image (list): List of candidates per image.
        dct_blocks_per_image (list): List of DCT block info per image.
        sample_image_paths (list): Original file paths.
        titles (list): Titles for each image.
        payloads (list): Payload bits for each image.

    Returns:
        filtered_candidates (list)
        filtered_dct_blocks (list)
        filtered_image_paths (list)
        filtered_titles (list)
        filtered_payloads (list)
    """
    filtered_candidates = []
    filtered_dct_blocks = []
    filtered_image_paths = []
    filtered_titles = []
    filtered_payloads = []

    for idx, candidates in enumerate(embedding_candidates_per_image):
        if len(candidates) > 0:
            filtered_candidates.append(candidates)
            filtered_dct_blocks.append(dct_blocks_per_image[idx])
            filtered_image_paths.append(sample_image_paths[idx])
            filtered_titles.append(titles[idx])
            filtered_payloads.append(payloads[idx])

    print(f"\nFiltered {len(embedding_candidates_per_image) - len(filtered_candidates)} images with zero embedding opportunities.")
    print(f"Remaining images for embedding: {len(filtered_candidates)}")

    return filtered_candidates, filtered_dct_blocks, filtered_image_paths, filtered_titles, filtered_payloads

# Run the filtering step
(
    filtered_candidates_per_image,
    filtered_dct_blocks_per_image,
    filtered_sample_image_paths,
    filtered_titles,
    filtered_payloads
) = filter_images_with_candidates(
    embedding_candidates_per_image,
    dct_blocks_per_image,
    sample_image_paths,
    titles,
    payloads
)


### Code Block #4: Sanity Check - Visualize Candidate Counts Per Image

In [ ]:
def visualize_filtered_embedding_candidates(filtered_candidates_per_image, filtered_titles):
    """
    Plot number of embedding candidate blocks for filtered images.

    Args:
        filtered_candidates_per_image (list): Filtered candidate data.
        filtered_titles (list): Titles for remaining images.
    """
    num_blocks = [len(candidates) for candidates in filtered_candidates_per_image]

    plt.figure(figsize=(8, 4))
    plt.bar(range(len(num_blocks)), num_blocks, tick_label=filtered_titles)
    plt.title("Filtered: Embedding Candidate Blocks Per Image")
    plt.xlabel("Image")
    plt.ylabel("Embedding Candidate Blocks")
    plt.xticks(rotation=45)
    plt.show()

# Show filtered candidate block counts
visualize_filtered_embedding_candidates(filtered_candidates_per_image, filtered_titles)



## Step 9: Embed Payload Bits into the Selected DCT Coefficients


#### Step 9 - tasks


- Embed your **payload bits** into the **mid-mid DCT coefficients** of each good block,  
  for each image that has embedding opportunities.

- Use the **4-8 QnT bandwidth** to guide **safe embedding**.

- Modify coefficients **gently**:
    - **LSB modification**  
    - Minimal **±1 tweak**

- Keep track of **how many bits** we’ve embedded per image.

- Prep for **reconstruction** in the next step (**inverse DCT**).


### Code Block #1: The Embed Function (Revised and Cleaned Up)


In [ ]:
def embed_bits_in_dct_blocks(dct_blocks_info, embedding_candidates, payload_bits, low_threshold=4, high_threshold=8):
    """
    Embed bits into the DCT coefficients of selected blocks for a single image.

    Args:
        dct_blocks_info (list): DCT blocks of the image (from Step 7).
        embedding_candidates (list): Candidate blocks for embedding (from Step 8).
        payload_bits (numpy array): The payload bitstream.
        low_threshold (int): Minimum Q value for embedding.
        high_threshold (int): Maximum Q value for embedding.

    Returns:
        modified_dct_blocks_info (list): Modified DCT blocks with embedded bits.
        num_embedded_bits (int): Number of bits successfully embedded.
    """
    bit_idx = 0
    max_bits = len(payload_bits)

    # Make a deep copy so we don't modify the originals
    import copy
    modified_dct_blocks_info = copy.deepcopy(dct_blocks_info)

    print(f"\nEmbedding up to {max_bits} bits in current image...")

    for block in embedding_candidates:
        block_position = block[0]['block_position']  # Same for all subblocks in this block group

        for subblock_info in block:
            subblock_idx = subblock_info['subblock_idx']
            candidates = subblock_info['candidates']

            # Reference to the actual DCT block we're modifying
            dct_block = modified_dct_blocks_info[
                next((idx for idx, blk in enumerate(dct_blocks_info) if blk['block_position'] == block_position))
            ]['dct_blocks'][subblock_idx]

            for candidate in candidates:
                y, x = candidate['freq']
                q_value = candidate['q_value']

                # Only embed if QnT value is within bandwidth
                if low_threshold <= q_value <= high_threshold:
                    if bit_idx >= max_bits:
                        print("All payload bits embedded!")
                        return modified_dct_blocks_info, bit_idx

                    for channel in range(3):  # Embed into R, G, B channels
                        coeff_int = int(round(dct_block[y, x, channel]))

                        # Modify the LSB of the coefficient to embed the payload bit
                        if coeff_int >= 0:
                            new_coeff = (coeff_int & ~1) | payload_bits[bit_idx]
                        else:
                            new_coeff = -((abs(coeff_int) & ~1) | payload_bits[bit_idx])

                        # Update the DCT block with the new coefficient
                        dct_block[y, x, channel] = float(new_coeff)

                        # Debug output (optional, can comment out later)
                        print(f"Embedded bit {payload_bits[bit_idx]} at block {block_position}, subblock {subblock_idx}, freq ({y},{x}), channel {channel}, Q={q_value}")

                        # Move to the next bit
                        bit_idx += 1

                        # Stop if we've embedded all bits
                        if bit_idx >= max_bits:
                            break

                else:
                    print(f"Skipping freq ({y},{x}) due to Q={q_value}")

            # Stop if we've embedded all bits
            if bit_idx >= max_bits:
                break

        if bit_idx >= max_bits:
            break

    print(f"Finished embedding {bit_idx} bits in this image.")
    return modified_dct_blocks_info, bit_idx

### Code Block #2: Loop Through All Images and Embed Payloads

In [ ]:
# Store modified DCT blocks after embedding
modified_dct_blocks_per_image = []
embedded_bits_per_image = []

for idx in range(len(filtered_candidates_per_image)):
    image_title = filtered_titles[idx]
    payload_bits = filtered_payloads[idx]
    num_payload_bits = len(payload_bits)
    num_candidate_blocks = len(filtered_candidates_per_image[idx])
    capacity_estimate = num_candidate_blocks * 4  # Since we're assuming 4 bits per block, adjust if needed

    # Optional: if you originally used string_to_bits to create the payload, show the original string
    # If you have it still available, otherwise this is just for show
    try:
        message_payload_str = ''.join([chr(int("".join(map(str, payload_bits[i:i+8])), 2)) for i in range(0, len(payload_bits), 8)])
    except:
        message_payload_str = "[Not a valid string - likely random bits]"

    print("\n" + "=" * 60)
    print(f"=== Embedding in Image {idx+1}: {image_title} ===")
    print("=" * 60)
    print(f"Original Payload String: {message_payload_str}")
    print(f"Converted Payload Bits: {''.join(map(str, payload_bits.tolist()))}")
    print(f"Number of Bits in Payload: {num_payload_bits}")
    print(f"Available Candidate Blocks: {num_candidate_blocks}")
    print(f"Estimated Capacity (blocks x bits per block): {capacity_estimate} bits")
    print(f"Using Bandwidth Threshold: Low=4, High=8")
    print("=" * 60)

    modified_dct_blocks, num_embedded_bits = embed_bits_in_dct_blocks(
        dct_blocks_info=filtered_dct_blocks_per_image[idx],
        embedding_candidates=filtered_candidates_per_image[idx],
        payload_bits=payload_bits,
        low_threshold=4,
        high_threshold=8
    )

    modified_dct_blocks_per_image.append(modified_dct_blocks)
    embedded_bits_per_image.append(num_embedded_bits)

    print(f"\nDone Embedding in Image {idx+1}: {image_title}")
    print(f"Total Bits Embedded: {num_embedded_bits}/{num_payload_bits} bits")
    print("=" * 60 + "\n")


### Code Block #3: Visual Check / Summary of Embedding

In [ ]:
# Summary of embedding per image
for idx, num_bits in enumerate(embedded_bits_per_image):
    print(f"Image {idx+1}: {filtered_titles[idx]} - {num_bits} bits embedded.")


## Step 10: Inverse DCT and Reconstruct the Stego Images

### Code Block #1: Helper Function to Rebuild 16x16 Blocks from Modified DCT Blocks

In [ ]:
def dct_blocks_to_spatial_block(dct_blocks):
    """
    Perform inverse DCT on four 8x8 DCT sub-blocks and reconstruct a 16x16 block.

    Args:
        dct_blocks (list): List of four 8x8x3 DCT blocks (modified).

    Returns:
        block_16x16 (numpy array): The reconstructed 16x16x3 spatial block (uint8).
    """
    block_16x16 = np.zeros((16, 16, 3), dtype=np.float32)

    # Rebuild the 16x16 block from four 8x8 sub-blocks
    for i in range(2):  # rows
        for j in range(2):  # columns
            subblock_idx = i * 2 + j
            dct_sub_block = dct_blocks[subblock_idx]

            # Perform inverse DCT on each channel
            spatial_sub_block = np.zeros_like(dct_sub_block)
            for c in range(3):
                spatial_sub_block[:, :, c] = cv2.idct(dct_sub_block[:, :, c])

            # Place sub-block into the 16x16 block
            y_start = i * 8
            x_start = j * 8
            block_16x16[y_start:y_start+8, x_start:x_start+8, :] = spatial_sub_block

    # Clip and convert to uint8 for image format
    block_16x16 = np.clip(block_16x16, 0, 255).astype(np.uint8)

    return block_16x16


### Code Block #2: Reconstruct the Entire Image From Modified Blocks

In [ ]:
def reconstruct_image_from_dct_blocks(original_image, modified_dct_blocks_info, block_size=16):
    """
    Reconstruct the stego image from modified DCT blocks.

    Args:
        original_image (numpy array): The original RGB image (uint8).
        modified_dct_blocks_info (list): List of modified DCT block infos (with positions and DCT data).
        block_size (int): Size of the block (16x16 by default).

    Returns:
        reconstructed_image (numpy array): Reconstructed stego image (uint8).
    """
    # Make a copy of the original image to modify
    reconstructed_image = original_image.copy()

    for block_info in modified_dct_blocks_info:
        block_position = block_info['block_position']
        dct_blocks = block_info['dct_blocks']

        # Inverse DCT and rebuild the 16x16 block
        block_16x16 = dct_blocks_to_spatial_block(dct_blocks)

        # Replace the block in the reconstructed image
        y_start = block_position[0] * block_size
        x_start = block_position[1] * block_size

        reconstructed_image[y_start:y_start+block_size, x_start:x_start+block_size, :] = block_16x16

    return reconstructed_image


### Code Block #3: Reconstruct and Display Stego Images Side by Side

In [ ]:
def display_original_and_stego_images(filtered_sample_images, reconstructed_images, filtered_titles):
    """
    Display original and stego images side by side for comparison.

    Args:
        filtered_sample_images (list): List of original images (normalized RGB arrays).
        reconstructed_images (list): List of reconstructed stego images (uint8).
        filtered_titles (list): Titles for each image.
    """
    num_images = len(filtered_sample_images)

    fig, axes = plt.subplots(2, num_images, figsize=(num_images * 4, 8))

    # If there's only one image, axes won't be 2D—reshape it manually
    if num_images == 1:
        axes = np.array(axes).reshape(2, 1)

    for i in range(num_images):
        # Original image (converted to uint8 for consistency)
        original_img = (filtered_sample_images[i] * 255).astype(np.uint8) if filtered_sample_images[i].max() <= 1.0 else filtered_sample_images[i]

        # Stego image
        stego_img = reconstructed_images[i]

        # Row 0: Original
        axes[0, i].imshow(original_img)
        axes[0, i].set_title(f"Original: {filtered_titles[i]}")
        axes[0, i].axis('off')

        # Row 1: Stego
        axes[1, i].imshow(stego_img)
        axes[1, i].set_title(f"Stego: {filtered_titles[i]}")
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()



### Code Block #4: Reconstruct Images and Run the Display

In [ ]:
def load_filtered_images(filtered_image_paths):
    """
    Load and preprocess filtered images from file paths.

    Args:
        filtered_image_paths (list): List of image paths.

    Returns:
        filtered_sample_images (list): List of loaded and preprocessed images.
    """
    filtered_sample_images = []

    for path in filtered_image_paths:
        img = Image.open(path).convert("RGB")
        img = img.resize((224, 224))  # Assuming you resized earlier
        img_array = np.array(img) / 255.0  # Normalize [0,1] as before
        filtered_sample_images.append(img_array)

    print(f"Loaded {len(filtered_sample_images)} filtered images.")
    return filtered_sample_images

# Run it
filtered_sample_images = load_filtered_images(filtered_sample_image_paths)


In [ ]:
reconstructed_images = []

for idx in range(len(filtered_dct_blocks_per_image)):
    print(f"Reconstructing Image {idx+1}: {filtered_titles[idx]}")

    # Get the original image for reference
    original_img = (filtered_sample_images[idx] * 255).astype(np.uint8) if filtered_sample_images[idx].max() <= 1.0 else filtered_sample_images[idx]

    # Reconstruct stego image
    stego_img = reconstruct_image_from_dct_blocks(
        original_image=original_img,
        modified_dct_blocks_info=modified_dct_blocks_per_image[idx]
    )

    reconstructed_images.append(stego_img)

# Display original and stego images for visual comparison
display_original_and_stego_images(filtered_sample_images, reconstructed_images, filtered_titles)


## Step 11: Evaluate Image Quality - PSNR & SSIM & MSE

### Code Block #1: Setup PSNR & SSIM Functions

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import numpy as np

def evaluate_stego_images(filtered_sample_images, reconstructed_images, filtered_titles):
    """
    Evaluate PSNR, SSIM, and MSE between original and stego images.

    Args:
        filtered_sample_images (list): List of original images (normalized RGB arrays).
        reconstructed_images (list): List of reconstructed stego images (uint8).
        filtered_titles (list): Titles for each image.

    Returns:
        evaluation_results (list): List of dicts with PSNR, SSIM, and MSE per image.
    """
    evaluation_results = []

    for idx in range(len(filtered_sample_images)):
        # Prepare original and stego images in uint8
        original_img = (filtered_sample_images[idx] * 255).astype(np.uint8) if filtered_sample_images[idx].max() <= 1.0 else filtered_sample_images[idx]
        stego_img = reconstructed_images[idx]

        # PSNR (computed on full RGB images)
        psnr_value = psnr(original_img, stego_img, data_range=255)

        # SSIM (computed on full RGB images)
        ssim_value = ssim(
            original_img,
            stego_img,
            channel_axis=-1,   # Instead of multichannel=True
            data_range=255
        )

        # MSE (Mean Squared Error)
        mse_value = np.mean((original_img.astype("float32") - stego_img.astype("float32")) ** 2)

        result = {
            "Image": filtered_titles[idx],
            "PSNR (dB)": round(psnr_value, 2),
            "SSIM": round(ssim_value, 4),
            "MSE": round(mse_value, 4)
        }

        evaluation_results.append(result)

        print(f"Evaluated {filtered_titles[idx]} --> PSNR: {result['PSNR (dB)']} dB, SSIM: {result['SSIM']}, MSE: {result['MSE']}")

    return evaluation_results


### Code Block #2: Run Evaluation on Your Images

In [ ]:
evaluation_results = evaluate_stego_images(filtered_sample_images, reconstructed_images, filtered_titles)


### Code Block #3: Visualize The Scores


In [ ]:
import pandas as pd

def display_evaluation_results(evaluation_results):
    """
    Display the evaluation results in a table.

    Args:
        evaluation_results (list): List of dicts with evaluation data.
    """
    df = pd.DataFrame(evaluation_results)
    display(df)

# Show PSNR and SSIM results table
display_evaluation_results(evaluation_results)


###   Code Block: Save Stego Images (PNG or JPEG)

In [ ]:
def save_original_and_stego_images(filtered_sample_images, reconstructed_images, filtered_titles, save_dir="image_outputs", format="png", jpeg_quality=90):
    """
    Save both original and stego images to a directory in the specified format.

    Args:
        filtered_sample_images (list): List of original images (normalized RGB arrays).
        reconstructed_images (list): List of stego images (uint8).
        filtered_titles (list): Image titles to use as filenames.
        save_dir (str): Directory to save images.
        format (str): Image format ('png' for lossless, 'jpeg' for lossy).
        jpeg_quality (int): Quality for JPEG images (1-100).
    """
    import os
    from PIL import Image

    originals_dir = os.path.join(save_dir, "originals")
    stegos_dir = os.path.join(save_dir, "stegos")

    # Create directories if they don't exist
    os.makedirs(originals_dir, exist_ok=True)
    os.makedirs(stegos_dir, exist_ok=True)

    for idx, (original_img, stego_img) in enumerate(zip(filtered_sample_images, reconstructed_images)):
        # Prepare original image (convert back to uint8 if needed)
        original_img_uint8 = (original_img * 255).astype(np.uint8) if original_img.max() <= 1.0 else original_img

        # Convert to PIL Images
        orig_pil = Image.fromarray(original_img_uint8)
        stego_pil = Image.fromarray(stego_img)

        # Get base filename (strip file extension)
        base_name = os.path.splitext(filtered_titles[idx])[0]

        # File paths
        orig_file = os.path.join(originals_dir, f"{base_name}_original.{format}")
        stego_file = os.path.join(stegos_dir, f"{base_name}_stego.{format}")

        # Save original
        if format.lower() == "jpeg":
            orig_pil.save(orig_file, "JPEG", quality=jpeg_quality)
        else:
            orig_pil.save(orig_file)

        # Save stego
        if format.lower() == "jpeg":
            stego_pil.save(stego_file, "JPEG", quality=jpeg_quality)
        else:
            stego_pil.save(stego_file)

        print(f"Saved Original Image: {orig_file}")
        print(f"Saved Stego Image: {stego_file}")

# Run it - save PNGs (or JPEGs if you like)
save_original_and_stego_images(filtered_sample_images, reconstructed_images, filtered_titles, save_dir="image_outputs", format="png")  # or "jpeg"
